# cosegmentation — setup no Colab

Ordem de execução: (1) checar GPU, (2) montar Drive, (3) clonar/atualizar o repo, (4) instalar dependências, (5) instalar SAM2 + baixar checkpoint, (6) validar dataset, (7) smoke test rápido antes de rodar qualquer sweep pesado.

**Antes de rodar qualquer coisa longa**: confirme em Runtime > Change runtime type que o acelerador está em GPU (T4 no Colab free/Pro; A100/L4 se disponível no Pro/Pro+).

## 1. Checar GPU disponível

In [1]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
print("CUDA version (torch):", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Sat Jul 25 00:54:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Montar o Google Drive

Tudo que precisa sobreviver ao fim da sessão (dataset, features, resultados de ablação) deve ser lido/escrito dentro de `/content/drive/MyDrive/...`, nunca só no disco local da VM (`/content/...` fora do Drive some quando a sessão encerra).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Ajuste para o caminho real dentro do seu Drive, depois de subir os dados.
DRIVE_ROOT = "/content/drive/MyDrive/cosegmentation"
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("DRIVE_ROOT:", DRIVE_ROOT)

Mounted at /content/drive
DRIVE_ROOT: /content/drive/MyDrive/cosegmentation


## 3. Clonar (ou atualizar) o repositório

Repositório privado: o token é lido dos **Secrets** do Colab (ícone de chave 🔑 na barra lateral), nunca digitado ou hardcoded aqui. Antes de rodar esta célula: adicione um secret chamado `GITHUB_TOKEN` com um Personal Access Token (fine-grained, escopo só nesse repo, permissão `Contents: Read` já basta para clonar) e habilite o acesso dele a este notebook.

Clonamos DENTRO do Drive, não no disco efêmero — assim o repo persiste entre sessões e você não precisa reclonar toda vez.

In [3]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER = "lucaslattari"          # AJUSTAR
GITHUB_REPO = "cosegmentation"        # AJUSTAR se o nome do repo for outro

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
REPO_DIR = f"/content/{GITHUB_REPO}"   # disco local do Colab, NÃO dentro do Drive

import os
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 "$REPO_URL" "$REPO_DIR"
else:
    print("Repo já existe nesta sessão, atualizando...")
    !cd "$REPO_DIR" && git pull

%cd $REPO_DIR
!git status

del GITHUB_TOKEN

Cloning into '/content/cosegmentation'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 117 (delta 6), reused 114 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 22.23 MiB | 37.07 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/cosegmentation
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


## 4. Instalar torch/torchvision/torchaudio (build exata do ambiente local)

Instalamos a MESMA build (`+cu126`) usada no `coseg_wsl`, via índice oficial do PyTorch, em vez de confiar no torch pré-instalado do Colab (que pode ser outra versão/build). Isso é mais lento que usar o que já vem pronto, mas garante que o comportamento numérico/CUDA seja o mesmo da sua máquina local — importante para resultados de ablação comparáveis.

In [4]:
!pip install -q \
  torch==2.12.0+cu126 \
  torchvision==0.27.0+cu126 \
  torchaudio==2.11.0+cu126 \
  --index-url https://download.pytorch.org/whl/cu126

import torch
print("torch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
assert torch.__version__.startswith("2.12.0"), "Versão instalada não bate com o esperado — confira o índice de wheels."

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 34.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.8/289.8 MB 76.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 45.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 218.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 80.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 176.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 93.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 61.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 135.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 87.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 120.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 170.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

AssertionError: Versão instalada não bate com o esperado — confira o índice de wheels.

## 5. Instalar as demais dependências + SAM2 (commit fixado)

O `requirements.txt` já cobre as libs gerais. SAM2 é instalado separadamente, fixado no MESMO commit usado no `coseg_wsl`, para reprodutibilidade — pule a linha do SAM2 se estiver rodando só `e1_sweep_features.sh` ou os diagnósticos de E1 (ele não é necessário até você chegar em E4).

In [ ]:
!pip install -q -r requirements.txt

# SAM2 — mesmo commit do ambiente local (ver requirements_local_exact.txt).
# Comente esta linha se ainda não for rodar E4/E5.
!pip install -q "git+https://github.com/facebookresearch/sam2.git@2b90b9f5ceec907a1c18123530e92e794ad901a4"

# Checkpoint sam2.1-hiera-large (o mesmo usado em produção pelo run_pipeline.sh).
import os
CKPT_DIR = f"{DRIVE_ROOT}/sam2_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
%cd $CKPT_DIR
!wget -nc https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

%cd $REPO_DIR

## 6. Validar dataset

Confirma que `datasets/icoseg/{images,ground_truth}` está acessível — subido para o Drive previamente (ver observação abaixo) ou copiado do próprio repo, se você decidiu versionar.

In [5]:
import os

DATASET_ROOT = f"{DRIVE_ROOT}/datasets/icoseg"   # continua no Drive, como você já subiu

for sub in ["images", "ground_truth"]:
    p = os.path.join(DATASET_ROOT, sub)
    exists = os.path.isdir(p)
    n = len(os.listdir(p)) if exists else 0
    print(f"{p}: {'OK' if exists else 'AUSENTE'} ({n} itens)")

/content/drive/MyDrive/cosegmentation/datasets/icoseg/images: AUSENTE (0 itens)
/content/drive/MyDrive/cosegmentation/datasets/icoseg/ground_truth: AUSENTE (0 itens)


## 7. Smoke test — antes de rodar qualquer sweep pesado

Roda E1 numa única classe pequena, com `--max_images` baixo, só para confirmar que o ambiente inteiro (torch+CUDA, imports, paths) está funcional antes de disparar horas de execução.

In [ ]:
!python pipeline/step1_feature_extraction.py \
  --collection "Panda" \
  --output /tmp/smoke_test_panda \
  --img_size 518 --resize keep_ar --feature_mode l2 \
  --max_images 3

## 8. Rodando o sweep de verdade — em pedaços, salvando direto no Drive

**Aponte `-o` para dentro do Drive**, não para `/content/...` fora dele, senão qualquer coisa gerada some se a sessão cair. Como o script já suporta `RESUME=1` (default) e marcadores `.done` por execução, rodar o mesmo comando de novo depois de uma queda de sessão retoma de onde parou — não precisa recomeçar do zero.

Se a sessão do Colab tende a cair antes do sweep terminar, considere quebrar por `-k` (regex de classe) ou por `MODELS_STR` (um modelo pesado como `vitl14` por vez) em vez de um único comando gigante.

In [ ]:
OUT_DIR = f"{DRIVE_ROOT}/ablation_results/e1_sweep_selected"

!MODELS_STR="dinov2_vits14 dinov2_vitb14 dinov2_vitl14" \
 IMG_SIZES_STR="448 518 672" \
 FEATURE_MODES_STR="raw l2 pca128_l2 pca256_l2" \
 RESIZE_MODES_STR="keep_ar square" \
 GT_INDEX="$REPO_DIR/gt_index.json" \
 bash ablation/e1/e1_sweep_features.sh \
   -j "$REPO_DIR/icoseg_index.json" \
   -s "$REPO_DIR/pipeline/step1_feature_extraction.py" \
   -a "$REPO_DIR/ablation/e1/e1_analyze_features.py" \
   -o "$OUT_DIR" \
   -g "$REPO_DIR/gt_index.json"